# **DATA BALANCING NOTEBOOK**

## **0. Library import**

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.utils import save_data
from src.config import BRFSS_CLEANED_FILE_PATH, LOG_DIR, TRAINING_DIR, \
    RANDOM_OVER_SAMPLING_DATA_FILE_PATH, SMOTE_DATA_FILE_PATH, ADASYN_DATA_FILE_PATH, \
    RANDOM_UNDER_SAMPLING_DATA_FILE_PATH, TOMEK_LINKS_DATA_FILE_PATH, EDITED_NEIGHBORS_DATA_FILE_PATH, \
    SMOTE_TOMEK_LINKS_DATA_FILE_PATH, SMOTE_ENN_DATA_FILE_PATH, ADASYN_TOMEK_DATA_FILE_PATH, \
    TESTING_DATA_FILE_PATH, TRAIN_SIZE, N_JOBS, OVER_SAMPLING_STRATEGY, UNDER_SAMPLING_STRATEGY
from src.features import DiabetesFeatureEngineering
from src.balancing import OverSamplingBalancer, UnderSamplingBalancer, HybridSamplingBalancer

## **1. Load data**

In [3]:
# Load preprocessed BRFSS dataset for data balancing
# Contains imbalanced classes: No diabetes (81.8%), Pre-diabetes (2.4%), Diabetes (15.8%)
df = pd.read_csv(filepath_or_buffer=BRFSS_CLEANED_FILE_PATH)
df.drop("Year", axis=1, inplace=True)
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,DiagnosedHeartAttack,CoronaryHeartDisease,COPD,AlcoholDays,LastCheckup,HasPersonalDoctor,CholesterolMeds,MaritalStatus,EmploymentStatus,CannotAffordDoctor
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,2.0,1.0,1.0,3.0,7.0,0.0
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,NaN,1.0,7.0,0.0
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,NaN,3.0,8.0,0.0
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,2.0,1.0,1.0,NaN,3.0,8.0,0.0
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,NaN,1.0,7.0,0.0


## **2. Feature engineering**

In [4]:
# Apply a complete feature engineering pipeline:
# - Remove duplicates and handle outliers
# - Create composite scores (Health, Risk, Lifestyle, Cardio)
# - Feature selection based on a correlation threshold (>0.1)
diabetes_feature_engineering = DiabetesFeatureEngineering(
    logger_name="data_balancing.feature_engineering_pipeline", 
    log_file=f"{LOG_DIR}/4_data_balancing.log"
)
processed_df, encoders, selected_features = diabetes_feature_engineering.process_all(df=df)
processed_df.shape

2025-09-28 04:39:30,975 - [data_balancing.feature_engineering_pipeline] - INFO - Enhanced DiabetesFeatureEngineering initialized successfully
2025-09-28 04:39:30,976 - [data_balancing.feature_engineering_pipeline] - INFO - ================================================================================
2025-09-28 04:39:30,976 - [data_balancing.feature_engineering_pipeline] - INFO - STARTING ENHANCED FEATURE ENGINEERING PIPELINE
2025-09-28 04:39:30,977 - [data_balancing.feature_engineering_pipeline] - INFO - ================================================================================
2025-09-28 04:39:30,978 - [data_balancing.feature_engineering_pipeline] - INFO - Initial dataset shape: (863745, 33)
2025-09-28 04:39:30,979 - [data_balancing.feature_engineering_pipeline] - INFO - Starting null values removal process...
2025-09-28 04:39:31,020 - [data_balancing.feature_engineering_pipeline] - INFO - Found 481755 null values across 23 columns
2025-09-28 04:39:31,021 - [data_balancing.fe

(449205, 34)

In [5]:
# Save selected features
save_data(
    path=f"{TRAINING_DIR}/selected_features.pkl", 
    data=selected_features
)
# Save encoders
save_data(
    path=f"{TRAINING_DIR}/encoders.pkl", 
    data=encoders
)

In [6]:
# Display the original class imbalance before balancing
# Class 0 (No diabetes): ~82% | Class 1 (Pre-diabetes): ~2% | Class 2 (Diabetes):
processed_df["Diabetes"].value_counts()

Diabetes
0.0    354270
2.0     82952
1.0     11983
Name: count, dtype: int64

In [7]:
# Get features and target
X = processed_df.drop(columns=["Diabetes"])
y = processed_df["Diabetes"]

In [8]:
# Split training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    train_size=TRAIN_SIZE, 
    random_state=42, 
    stratify=y
)

In [9]:
# Export testing data
save_data(
    path=TESTING_DATA_FILE_PATH,
    data={
        "X": X_test,
        "y": y_test
    }
)

In [10]:
y_train.value_counts()

Diabetes
0.0    283416
2.0     66362
1.0      9586
Name: count, dtype: int64

## **3. Data balancing**

### **3.1 Oversampling**

In [11]:
# Initialize OverSamplingBalancer
over_sampling_balancer = OverSamplingBalancer(
    logger_name="data_balancing.oversampling_balancer", 
    log_file=f"{LOG_DIR}/4_data_balancing.log"
)

2025-09-28 04:39:36,259 - [data_balancing.oversampling_balancer] - INFO - OverSamplingBalancer initialized successfully


#### **3.1.1 Random Oversampling**

In [12]:
# Apply Random Oversampling method with sampling strategy {1.0: 50000,2.0: 100000} on training data
ros_X_train, ros_y_train = over_sampling_balancer.apply_random_oversampling(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-28 04:39:36,269 - [data_balancing.oversampling_balancer] - INFO - Starting Random Over Sampling process...
2025-09-28 04:39:36,274 - [data_balancing.oversampling_balancer] - INFO - Original class distribution:
2025-09-28 04:39:36,276 - [data_balancing.oversampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 04:39:36,277 - [data_balancing.oversampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 04:39:36,278 - [data_balancing.oversampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 04:39:36,480 - [data_balancing.oversampling_balancer] - INFO - New class distribution after Random Over Sampling:
2025-09-28 04:39:36,480 - [data_balancing.oversampling_balancer] - INFO -   - Class 0.0: 283416 samples (+0)
2025-09-28 04:39:36,481 - [data_balancing.oversampling_balancer] - INFO -   - Class 1.0: 50000 samples (+40414)
2025-09-28 04:39:36,481 - [data_balancing.oversampling_balancer] - INFO -   - Class 2.0: 100000 samples (+33638)
2025-09-28 04:

In [13]:
# Verify balanced distribution after resampling
ros_y_train.value_counts()

Diabetes
0.0    283416
2.0    100000
1.0     50000
Name: count, dtype: int64

In [14]:
# Save Random Oversampling balanced dataset for model training
save_data(
    path=RANDOM_OVER_SAMPLING_DATA_FILE_PATH,
    data={
        "X": ros_X_train,
        "y": ros_y_train,
    }
)

#### **3.1.2 SMOTE**

In [15]:
# Apply SMOTE method with sampling strategy {1.0: 50000,2.0: 100000} on training data
smote_X_train, smote_y_train = over_sampling_balancer.apply_smote(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-28 04:39:36,616 - [data_balancing.oversampling_balancer] - INFO - Starting SMOTE process...
2025-09-28 04:39:36,621 - [data_balancing.oversampling_balancer] - INFO - Original class distribution:
2025-09-28 04:39:36,622 - [data_balancing.oversampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 04:39:36,622 - [data_balancing.oversampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 04:39:36,622 - [data_balancing.oversampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 04:39:43,693 - [data_balancing.oversampling_balancer] - INFO - New class distribution after SMOTE:
2025-09-28 04:39:43,694 - [data_balancing.oversampling_balancer] - INFO -   - Class 0.0: 283416 samples (+0 synthetic)
2025-09-28 04:39:43,695 - [data_balancing.oversampling_balancer] - INFO -   - Class 1.0: 50000 samples (+40414 synthetic)
2025-09-28 04:39:43,696 - [data_balancing.oversampling_balancer] - INFO -   - Class 2.0: 100000 samples (+33638 synthetic)
2025-09-28 04:

In [16]:
# Verify balanced distribution after resampling
smote_y_train.value_counts()

Diabetes
0.0    283416
2.0    100000
1.0     50000
Name: count, dtype: int64

In [17]:
# Save SMOTE balanced dataset for model training
save_data(
    path=SMOTE_DATA_FILE_PATH,
    data={
        "X": smote_X_train,
        "y": smote_y_train,
    }
)

#### **3.1.3 ADASYN**

In [18]:
# Apply ADASYN method with sampling strategy {1.0: 50000,2.0: 100000} on training data
adasyn_X_train, adasyn_y_train = over_sampling_balancer.apply_adasyn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-28 04:39:43,871 - [data_balancing.oversampling_balancer] - INFO - Starting ADASYN over-sampling process...
2025-09-28 04:39:43,879 - [data_balancing.oversampling_balancer] - INFO - Original class distribution:
2025-09-28 04:39:43,880 - [data_balancing.oversampling_balancer] - INFO -  - Class 0.0: 283416 samples
2025-09-28 04:39:43,882 - [data_balancing.oversampling_balancer] - INFO -  - Class 1.0: 9586 samples
2025-09-28 04:39:43,883 - [data_balancing.oversampling_balancer] - INFO -  - Class 2.0: 66362 samples
2025-09-28 04:40:52,335 - [data_balancing.oversampling_balancer] - INFO - New class distribution after ADASYN:
2025-09-28 04:40:52,336 - [data_balancing.oversampling_balancer] - INFO -  - Class 0.0: 283416 samples (+0 synthetic)
2025-09-28 04:40:52,336 - [data_balancing.oversampling_balancer] - INFO -  - Class 1.0: 49354 samples (+39768 synthetic)
2025-09-28 04:40:52,337 - [data_balancing.oversampling_balancer] - INFO -  - Class 2.0: 96460 samples (+30098 synthetic)
2025-

In [19]:
# Verify balanced distribution after resampling
adasyn_y_train.value_counts()

Diabetes
0.0    283416
2.0     96460
1.0     49354
Name: count, dtype: int64

In [20]:
# Save ADASYN balanced dataset for model training
save_data(
    path=ADASYN_DATA_FILE_PATH,
    data={
        "X": adasyn_X_train,
        "y": adasyn_y_train,
    }
)

### **3.2 Undersampling**

In [21]:
# Initialize UnderSamplingBalancer
under_sampling_balancer = UnderSamplingBalancer(
    logger_name="data_balancing.undersampling_balancer", 
    log_file=f"{LOG_DIR}/4_data_balancing.log"
)

2025-09-28 04:40:52,549 - [data_balancing.undersampling_balancer] - INFO - UnderSamplingBalancer initialized successfully


#### **3.2.1 Random Undersampling**

In [22]:
# Apply Random Undersampling method with {0: 100000, 2: 100000, 1: 16890} on training data
rus_X_train, rus_y_train = under_sampling_balancer.apply_random_undersampling(
    X=X_train, 
    y=y_train,
    sampling_strategy=UNDER_SAMPLING_STRATEGY
)

2025-09-28 04:40:52,560 - [data_balancing.undersampling_balancer] - INFO - Starting Random Under Sampling process...
2025-09-28 04:40:52,570 - [data_balancing.undersampling_balancer] - INFO - Original class distribution:
2025-09-28 04:40:52,572 - [data_balancing.undersampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 04:40:52,573 - [data_balancing.undersampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 04:40:52,574 - [data_balancing.undersampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 04:40:52,744 - [data_balancing.undersampling_balancer] - INFO - New class distribution after Random Under Sampling:
2025-09-28 04:40:52,745 - [data_balancing.undersampling_balancer] - INFO -   - Class 0.0: 150000 samples (-133416)
2025-09-28 04:40:52,745 - [data_balancing.undersampling_balancer] - INFO -   - Class 1.0: 9586 samples (-0)
2025-09-28 04:40:52,746 - [data_balancing.undersampling_balancer] - INFO -   - Class 2.0: 66362 samples (-0)
2025-09-

In [23]:
# Verify balanced distribution after resampling
rus_y_train.value_counts()

Diabetes
0.0    150000
2.0     66362
1.0      9586
Name: count, dtype: int64

In [24]:
# Save Random Undersamplinng balanced dataset for model training
save_data(
    path=RANDOM_UNDER_SAMPLING_DATA_FILE_PATH,
    data={
        "X": rus_X_train,
        "y": rus_y_train,
    }
)

#### **3.2.2 TomekLinks**

In [25]:
# Apply Tomek Links method with sampling strategy 'auto' on training data
tomek_X_train, tomek_y_train = under_sampling_balancer.apply_tomek_links(
    X=X_train, 
    y=y_train, 
    n_jobs=N_JOBS
)

2025-09-28 04:40:52,874 - [data_balancing.undersampling_balancer] - INFO - Starting Tomek Links process...
2025-09-28 04:40:52,884 - [data_balancing.undersampling_balancer] - INFO - Original class distribution:
2025-09-28 04:40:52,885 - [data_balancing.undersampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 04:40:52,885 - [data_balancing.undersampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 04:40:52,887 - [data_balancing.undersampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 04:45:31,096 - [data_balancing.undersampling_balancer] - INFO - New class distribution after Tomek Links:
2025-09-28 04:45:31,097 - [data_balancing.undersampling_balancer] - INFO -   - Class 0.0: 274180 samples (-9236 Tomek links)
2025-09-28 04:45:31,099 - [data_balancing.undersampling_balancer] - INFO -   - Class 1.0: 9586 samples (unchanged)
2025-09-28 04:45:31,100 - [data_balancing.undersampling_balancer] - INFO -   - Class 2.0: 57778 samples (-8584 Tomek lin

In [26]:
# Verify balanced distribution after resampling
tomek_y_train.value_counts()

Diabetes
0.0    274180
2.0     57778
1.0      9586
Name: count, dtype: int64

In [27]:
# Save Tomek Links balanced dataset for model training
save_data(
    path=TOMEK_LINKS_DATA_FILE_PATH,
    data={
        "X": tomek_X_train,
        "y": tomek_y_train,
    }
)

#### **3.2.3 Edited Nearest Neighbors**

In [28]:
# Apply Edited Nearest Neighbors method with sampling strategy 'auto' on training data
enn_X_train, enn_y_train = under_sampling_balancer.apply_edited_nearest_neighbours(
    X=X_train, 
    y=y_train,
    n_jobs=N_JOBS,
)

2025-09-28 04:45:31,241 - [data_balancing.undersampling_balancer] - INFO - Starting Edited Nearest Neighbours (ENN) under-sampling process...
2025-09-28 04:45:31,250 - [data_balancing.undersampling_balancer] - INFO - Original class distribution:
2025-09-28 04:45:31,251 - [data_balancing.undersampling_balancer] - INFO -  - Class 0.0: 283416 samples
2025-09-28 04:45:31,252 - [data_balancing.undersampling_balancer] - INFO -  - Class 1.0: 9586 samples
2025-09-28 04:45:31,254 - [data_balancing.undersampling_balancer] - INFO -  - Class 2.0: 66362 samples
2025-09-28 04:50:01,088 - [data_balancing.undersampling_balancer] - INFO - New class distribution after ENN:
2025-09-28 04:50:01,089 - [data_balancing.undersampling_balancer] - INFO -  - Class 0.0: 218788 samples (-64628 removed)
2025-09-28 04:50:01,089 - [data_balancing.undersampling_balancer] - INFO -  - Class 1.0: 9586 samples (-0 removed)
2025-09-28 04:50:01,090 - [data_balancing.undersampling_balancer] - INFO -  - Class 2.0: 25899 sampl

In [29]:
# Verify balanced distribution after resampling
enn_y_train.value_counts()

Diabetes
0.0    218788
2.0     25899
1.0      9586
Name: count, dtype: int64

In [30]:
# Save Edited Nearest Neighbors balanced dataset for model training
save_data(
    path=EDITED_NEIGHBORS_DATA_FILE_PATH,
    data={
        "X": enn_X_train,
        "y": enn_y_train,
    }
)

### **3.3 Hybrid**

In [31]:
# Initialize HybridSamplingBalancer
hybrid_sampling_balancer = HybridSamplingBalancer(
    logger_name="data_balancing.hybrid_sampling_balancer", 
    log_file=f"{LOG_DIR}/4_data_balancing.log"
)

2025-09-28 04:50:01,202 - [data_balancing.hybrid_sampling_balancer] - INFO - HybridSamplingBalancer initialized successfully


#### **3.3.1 SMOTE + Tomek Links**


In [32]:
# Apply SMOTE + Tomek Links method with sampling strategy 'auto' on training data
smote_tomek_X_train, smote_tomek_y_train = hybrid_sampling_balancer.apply_smote_tomek(
    X=X_train, 
    y=y_train,
    sampling_strategy=OVER_SAMPLING_STRATEGY,
    n_jobs=N_JOBS,
)

2025-09-28 04:50:01,214 - [data_balancing.hybrid_sampling_balancer] - INFO - Starting SMOTETomek hybrid sampling process...
2025-09-28 04:50:01,219 - [data_balancing.hybrid_sampling_balancer] - INFO - Original class distribution:
2025-09-28 04:50:01,221 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 04:50:01,222 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 04:50:01,223 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 04:56:54,962 - [data_balancing.hybrid_sampling_balancer] - INFO - New class distribution after SMOTETomek:
2025-09-28 04:56:54,963 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 0.0: 279388 samples (-4028 net)
2025-09-28 04:56:54,964 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 1.0: 49787 samples (+40201 net)
2025-09-28 04:56:54,965 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 2.0: 9611

In [33]:
# Verify balanced distribution after resampling
smote_tomek_y_train.value_counts()

Diabetes
0.0    279388
2.0     96111
1.0     49787
Name: count, dtype: int64

In [34]:
# # Save SMOTE + Tomek Links balanced dataset for model training
save_data(
    path=SMOTE_TOMEK_LINKS_DATA_FILE_PATH,
    data={
        "X": smote_tomek_X_train,
        "y": smote_tomek_y_train,
    }
)

#### **3.3.2 SMOTE + ENN**

In [35]:
# Apply SMOTE + ENN method with sampling strategy {1.0: 50000, 2.0: 100000} on training data
smoteen_X_train, smoteen_y_train = hybrid_sampling_balancer.apply_smote_enn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY, 
    n_jobs=N_JOBS
)

2025-09-28 04:56:55,120 - [data_balancing.hybrid_sampling_balancer] - INFO - Starting SMOTEENN hybrid sampling process...
2025-09-28 04:56:55,127 - [data_balancing.hybrid_sampling_balancer] - INFO - Original class distribution:
2025-09-28 04:56:55,128 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 04:56:55,129 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 04:56:55,131 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 05:03:48,784 - [data_balancing.hybrid_sampling_balancer] - INFO - New class distribution after SMOTEENN:
2025-09-28 05:03:48,785 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 0.0: 204391 samples (-79025 net)
2025-09-28 05:03:48,786 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 1.0: 45067 samples (+35481 net)
2025-09-28 05:03:48,787 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 2.0: 56031 s

In [36]:
# Verify balanced distribution after resampling
smoteen_y_train.value_counts()

Diabetes
0.0    204391
2.0     56031
1.0     45067
Name: count, dtype: int64

In [37]:
# Save SMOTE + ENN balanced dataset for model training
save_data(
    path=SMOTE_ENN_DATA_FILE_PATH,
    data={
        "X": smoteen_X_train,
        "y": smoteen_y_train,
    }
)

#### **3.3.3 ADASYN + Tomek Links**


In [38]:
# Apply ADASYN + Tomek Links method with sampling strategy {1.0: 50000,2.0: 100000} method on training data
adasyn_tomek_X_train, adasyn_tomek_y_train = hybrid_sampling_balancer.apply_adasyn_tomek(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY,
    n_jobs=N_JOBS,
)

2025-09-28 05:03:48,915 - [data_balancing.hybrid_sampling_balancer] - INFO - Starting ADASYN + TomekLinks hybrid sampling process...
2025-09-28 05:03:48,923 - [data_balancing.hybrid_sampling_balancer] - INFO - Original class distribution:
2025-09-28 05:03:48,924 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 0.0: 283416 samples
2025-09-28 05:03:48,925 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 1.0: 9586 samples
2025-09-28 05:03:48,926 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 2.0: 66362 samples
2025-09-28 05:11:38,673 - [data_balancing.hybrid_sampling_balancer] - INFO - New class distribution after ADASYN + TomekLinks:
2025-09-28 05:11:38,674 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 0.0: 281315 samples (-2101 net)
2025-09-28 05:11:38,675 - [data_balancing.hybrid_sampling_balancer] - INFO -   - Class 1.0: 49354 samples (+39768 net)
2025-09-28 05:11:38,675 - [data_balancing.hybrid_sampling_balancer] - INFO -  

In [39]:
# Verify balanced distribution after resampling
adasyn_tomek_y_train.value_counts()

Diabetes
0.0    281315
2.0     94491
1.0     49354
Name: count, dtype: int64

In [40]:
# Save ADASYN + Tomek Links balanced dataset for model training
save_data(
    path=ADASYN_TOMEK_DATA_FILE_PATH,
    data={
        "X": adasyn_tomek_X_train,
        "y": adasyn_tomek_y_train,
    }
)